In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
import torch
import os

# ======================
# 0. Disable W&B
# ======================
os.environ["WANDB_DISABLED"] = "true"

# ======================
# 1. Load CSV
# ======================
csv_path = "/kaggle/input/hp-all/hp_paraphrase_pairs_narrative_raw.csv"  # change path
df = pd.read_csv(csv_path)

if not {"input", "target"}.issubset(df.columns):
    raise ValueError(f"CSV must have 'input' and 'target' columns. Found: {list(df.columns)}")

# HuggingFace Dataset
dataset = Dataset.from_pandas(df[["input", "target"]])

# Split train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# ======================
# 2. Load Model & Tokenizer
# ======================
model_name = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

if torch.cuda.is_available():
    model = model.cuda()

# ======================
# 3. Preprocess Data (ADD TASK PREFIX)
# ======================
def preprocess(batch):
    inputs = tokenizer(
        ["hp_style: " + x for x in batch["input"]],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    targets = tokenizer(
        batch["target"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized = dataset.map(preprocess, batched=True, remove_columns=["input", "target"])

# ======================
# 4. Training Setup
# ======================
training_args = TrainingArguments(
    output_dir="./hp_t5_finetuned",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy="no",          # 🚨 disables checkpoint saving
    logging_dir="./logs",
    logging_steps=10,
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
)

# ======================
# 5. Train
# ======================
trainer.train()

# ======================
# 6. Save & Reload Model
# ======================
trainer.save_model("./hp_t5_finetuned")
tokenizer.save_pretrained("./hp_t5_finetuned")

tokenizer = T5Tokenizer.from_pretrained("./hp_t5_finetuned")
model = T5ForConditionalGeneration.from_pretrained("./hp_t5_finetuned")

if torch.cuda.is_available():
    model = model.cuda()

# ======================
# 7. Inference (WITH PREFIX)
# ======================
test_caption = "Harry looking snow out of clock tower"  # 👈 try your own
inputs = tokenizer("hp_style: " + test_caption, return_tensors="pt")

if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.no_grad():
    output = model.generate(**inputs, max_length=100, num_beams=4, early_stopping=True)

print("Original:", test_caption)
print("Styled  :", tokenizer.decode(output[0], skip_special_tokens=True))


2025-09-25 15:31:07.683994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758814267.872194      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758814267.932396      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/58288 [00:00<?, ? examples/s]

Map:   0%|          | 0/6477 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
10,30.755500
20,21.845100
30,13.027600
40,5.737300
50,4.497900
60,4.068500
70,3.209000
80,2.436400
90,1.756400
100,1.173700


Original: Harry looking snow out of clock tower
Styled  : Harry looked up at the sky.
